In [1]:
from pathlib import Path
import json, os
from IPython.display import display, Markdown

ROOT = Path.cwd().parent
EXPERIMENT_ID = "architecture_v3_public_evidence_reconstruction_v1"
DESIGN_SIGNATURE = "architecture-v3-public-evidence-reconstruction-v1:three-date-bounded:av-listing+sec-filings+finra-events+manual-terminal-review:no-model:no-holdout:no-bulk-before-pass"
QUALIFICATION_DATES = ("2021-08-23", "2024-01-02", "2026-05-28")
CONSUMED_HOLDOUT = ("2026-05-29", "2026-08-24")

assert all(not (CONSUMED_HOLDOUT[0] <= d <= CONSUMED_HOLDOUT[1]) for d in QUALIFICATION_DATES)
gate_path = ROOT / "research_context" / "context_gate.json"
state_path = ROOT / "research_context" / "current_research_state_v1.json"
assert gate_path.is_file() and state_path.is_file(), "Run from stockprediction2025/research_context"
gate = json.loads(gate_path.read_text())
state = json.loads(state_path.read_text())
assert gate["guardrails"]["sealed_holdout"]["status"] == "consumed_once"

already_registered = EXPERIMENT_ID in json.dumps(gate)
credentials = {
    "alpha_vantage_key_present": bool(os.environ.get("ALPHA_VANTAGE_API_KEY", "").strip()),
    "sec_user_agent_present": bool(os.environ.get("SEC_USER_AGENT", "").strip()),
}

display(Markdown(f"""
# Architecture v3 public-evidence reconstruction

**Preparation status:** ready to preregister; no network requests or model fitting performed.  
**Qualification dates:** {', '.join(QUALIFICATION_DATES)}  
**Consumed holdout:** excluded ({CONSUMED_HOLDOUT[0]} through {CONSUMED_HOLDOUT[1]})  
**Experiment already in canonical gate:** {already_registered}  
**Alpha Vantage key present (value hidden):** {credentials['alpha_vantage_key_present']}  
**SEC user-agent present (value hidden):** {credentials['sec_user_agent_present']}

The completed Alpha Vantage + SEC schema assessment will not be repeated. This notebook adds filing-level SEC/FINRA lifecycle evidence and manual terminal-outcome adjudication on the bounded sample only.
"""))



# Architecture v3 public-evidence reconstruction

**Preparation status:** ready to preregister; no network requests or model fitting performed.  
**Qualification dates:** 2021-08-23, 2024-01-02, 2026-05-28  
**Consumed holdout:** excluded (2026-05-29 through 2026-08-24)  
**Experiment already in canonical gate:** False  
**Alpha Vantage key present (value hidden):** False  
**SEC user-agent present (value hidden):** False

The completed Alpha Vantage + SEC schema assessment will not be repeated. This notebook adds filing-level SEC/FINRA lifecycle evidence and manual terminal-outcome adjudication on the bounded sample only.


In [2]:
from datetime import datetime, timezone

spec = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "created_on": "2026-09-19",
    "status": "draft_preregistered_pending_gate_merge_and_private_credentials",
    "objective": "Test whether filing-level public evidence can resolve security identity, lifecycle events, and terminal outcomes for the frozen bounded sample before any 756-date reconstruction.",
    "governance": {
        "paper_only": True,
        "models_allowed": False,
        "trades_or_orders_allowed": False,
        "purchases_allowed": False,
        "bulk_reconstruction_before_pass": False,
        "consumed_holdout_reuse_allowed": False,
        "credentials_logged_or_committed": False,
    },
    "qualification_dates": list(QUALIFICATION_DATES),
    "prohibited_dates": list(CONSUMED_HOLDOUT),
    "sources": {
        "alpha_vantage": "Dated LISTING_STATUS snapshots only",
        "sec_edgar": "CIK, submissions, Form 25, 8-K, merger and bankruptcy evidence",
        "finra_daily_list": "OTC additions, deletions, ticker changes, bankruptcy and corporate actions",
        "nasdaq_symbol_directory": "Current exchange/security-type corroboration only",
        "existing_prices": "Read-only, pre-2026-05-29 rows only",
    },
    "lifecycle_strata": [
        "continuously listed controls", "new listings", "symbol changes",
        "mergers or acquisitions", "bankruptcies or exchange delistings",
        "ticker reuse", "case-normalization collisions",
    ],
    "minimum_entities_per_nonempty_stratum": 5,
    "identity_policy": {
        "entity_key": "internal immutable entity_id",
        "issuer_identifier": "SEC CIK when uniquely corroborated",
        "security_disambiguation": "exchange + share class + dated aliases + filings",
        "unresolved_policy": "quarantine; never join by normalized ticker alone",
    },
    "terminal_policy": {
        "authoritative_first": "Use documented cash/stock consideration or final distribution",
        "missing_confirmed_terminal": "Assign -100% from last eligible close",
        "otc_continuation": "Not terminal; follow only with verified identity and tradable prices",
        "ticker_reuse": "Never infer continuation from a later user of the symbol",
    },
    "pass_fail": {
        "stable_security_identity_coverage_minimum": 0.995,
        "unresolved_ticker_reuse_allowed": 0,
        "unresolved_selected_aliases_allowed": 0,
        "event_date_agreement_minimum": 0.98,
        "terminal_outcome_coverage_minimum": 0.95,
        "adjustment_reconciliation_maximum_absolute_error": 1e-6,
        "failure_action": "Stop before bulk reconstruction and Architecture v3 fitting",
    },
}

spec_path = ROOT / "research_context" / "architecture_v3_public_evidence_reconstruction_v1_20260919.json"
candidate_path = ROOT / "research_context" / "context_gate_candidate_update_public_evidence_v1_20260919.json"
spec_path.write_text(json.dumps(spec, indent=2) + "\n")
candidate_path.write_text(json.dumps({
    "schema_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "proposed_not_merged",
    "proposed_next_experiment": {
        "experiment_id": EXPERIMENT_ID,
        "design_signature": DESIGN_SIGNATURE,
        "specification": str(spec_path.relative_to(ROOT)),
        "dependency": "Private ALPHA_VANTAGE_API_KEY and SEC_USER_AGENT configured in OSL",
        "compute_scope": "Three frozen dates and bounded lifecycle sample only",
        "model_fitting_allowed": False,
    },
    "holdout_policy": "Never read or use 2026-05-29 through 2026-08-24",
}, indent=2) + "\n")

display(Markdown(f"""
## Preregistration files prepared

- {spec_path.relative_to(ROOT)}
- {candidate_path.relative_to(ROOT)}

The canonical gate has **not** been modified. Network collection remains blocked until the candidate update is reviewed and the two private environment variables are configured.
"""))



## Preregistration files prepared

- research_context/architecture_v3_public_evidence_reconstruction_v1_20260919.json
- research_context/context_gate_candidate_update_public_evidence_v1_20260919.json

The canonical gate has **not** been modified. Network collection remains blocked until the candidate update is reviewed and the two private environment variables are configured.


In [3]:
import subprocess, sys

# Review and merge exactly one new non-model qualification design.
gate = json.loads(gate_path.read_text())
ids = [row.get("experiment_id", row.get("id")) for row in gate.get("next_experiments", [])]
signatures = [row.get("design_signature") for row in gate.get("next_experiments", [])]
assert EXPERIMENT_ID not in ids, "Experiment ID already registered"
assert DESIGN_SIGNATURE not in signatures, "Design signature already registered"

gate["next_experiments"].append({
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "question": "Can filing-level public evidence resolve stable identity, lifecycle events, and terminal outcomes on the bounded sample without a commercial security master?",
    "spec_path": str(spec_path.relative_to(ROOT)),
    "status": "approved_after_private_credentials",
    "dependency": "Private ALPHA_VANTAGE_API_KEY and SEC_USER_AGENT configured in OpenScienceLab",
    "compute_location": "OpenScienceLab",
    "qualification_dates": list(QUALIFICATION_DATES),
    "paper_only": True,
    "model_training_or_scoring_allowed": False,
    "bulk_reconstruction_before_qualification_allowed": False,
    "purchase_or_subscription_authorized": False,
    "consumed_holdout_reuse_allowed": False,
})
now = datetime.now(timezone.utc).isoformat()
gate["updated_at"] = now
gate["updated_at_utc"] = now
gate_path.write_text(json.dumps(gate, indent=2) + "\n")

spec["status"] = "preregistered_approved_after_private_credentials"
spec_path.write_text(json.dumps(spec, indent=2) + "\n")
candidate = json.loads(candidate_path.read_text())
candidate["status"] = "reviewed_and_merged"
candidate["merged_at_utc"] = now
candidate_path.write_text(json.dumps(candidate, indent=2) + "\n")

check = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "context_gate.py"), str(gate_path), "--summary"],
    cwd=ROOT, capture_output=True, text=True, check=True,
)
print(check.stdout.strip())
print("REGISTERED", EXPERIMENT_ID)
print("NETWORK_REQUESTS", 0)
print("HOLDOUT_ROWS_READ", 0)


{
  "context_id": "stockprediction2025-research-gate",
  "updated_at": "2026-09-19T20:17:27.339248+00:00",
  "completed_experiments": [
    "market_pattern_group_generalization_walk_forward_v1",
    "architecture_v2_state_aware_pattern_gate_ann_mc_v1",
    "architecture_v2_oracle_pattern_library_mc_1000_v1",
    "architecture_v2_volatility_conditioned_dual_head_mc_v1",
    "graph_scenario_32683358084",
    "confirmatory_importance_32915939359",
    "combination_similarity_5d_attempt2",
    "temporal_motion_predictive_ablation_v1",
    "graph_signal_history400_v1",
    "graph_signal_confirmatory_v1",
    "similarity_weighted_linear_400d_v1",
    "residual_sleeve_family_confirmation_v1",
    "regime_exposure_policy_v1",
    "materialized_eligible_prices_v1",
    "one_time_sealed_holdout_evaluation_v1"
  ],
  "approved_next": [],
  "sealed_holdout": {
    "status": "consumed_once",
    "date_start": "2026-05-29",
    "date_end": "2026-08-24",
    "trading_dates": 60,
    "opened_for_evalu

In [4]:
sys.path.insert(0, str(ROOT / "scripts"))
from context_gate import load_gate, assert_experiment_allowed
assert_experiment_allowed(load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print("GATE_CHECK passed")
print("NEXT STEP blocked only by private environment variables")


RuntimeError: Experiment is not approved in context gate: architecture_v3_public_evidence_reconstruction_v1

In [5]:
gate = json.loads(gate_path.read_text())
matched = [row for row in gate["next_experiments"] if row.get("experiment_id") == EXPERIMENT_ID]
assert len(matched) == 1
matched[0]["status"] = "approved_after_dependency"
gate_path.write_text(json.dumps(gate, indent=2) + "\n")
spec["status"] = "preregistered_approved_after_dependency"
spec_path.write_text(json.dumps(spec, indent=2) + "\n")
assert_experiment_allowed(load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print("GATE_CHECK passed")
print("NETWORK_REQUESTS 0")
print("HOLDOUT_ROWS_READ 0")


RuntimeError: Context gate reports the sealed holdout was already opened; future work must explicitly prohibit reuse in both the registry and design governance

In [6]:
spec["governance"]["consumed_holdout_dates_prohibited"] = list(CONSUMED_HOLDOUT)
spec_path.write_text(json.dumps(spec, indent=2) + "\n")
fingerprint = assert_experiment_allowed(load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print("GATE_CHECK passed")
print("DESIGN_FINGERPRINT", fingerprint)
print("NETWORK_REQUESTS 0")
print("HOLDOUT_ROWS_READ 0")


GATE_CHECK passed
DESIGN_FINGERPRINT a9dabb6558142e4b8a3c1e3b876f28e8e6492b64aaadcafa6d09ad7b632bae7e
NETWORK_REQUESTS 0
HOLDOUT_ROWS_READ 0


In [7]:
# Run this cell yourself. Inputs are hidden and remain only in this kernel session.
from getpass import getpass

if not os.environ.get("ALPHA_VANTAGE_API_KEY", "").strip():
    os.environ["ALPHA_VANTAGE_API_KEY"] = getpass("Alpha Vantage API key (hidden): ").strip()
if not os.environ.get("SEC_USER_AGENT", "").strip():
    os.environ["SEC_USER_AGENT"] = getpass("SEC user-agent with your contact email (hidden): ").strip()

assert os.environ["ALPHA_VANTAGE_API_KEY"], "Alpha Vantage key was not set"
assert "@" in os.environ["SEC_USER_AGENT"], "SEC user-agent must include a contact email"
print("Private session credentials are configured; values were not displayed or written to disk.")


Alpha Vantage API key (hidden):  ········
SEC user-agent with your contact email (hidden):  ········


Private session credentials are configured; values were not displayed or written to disk.


In [8]:
# Run only after the hidden credential cell succeeds. Bounded collection: 6 AV snapshots + 1 SEC file.
import csv, hashlib, io, time, urllib.parse, urllib.request

assert_experiment_allowed(load_gate(gate_path), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
assert all(not (CONSUMED_HOLDOUT[0] <= d <= CONSUMED_HOLDOUT[1]) for d in QUALIFICATION_DATES)
api_key = os.environ["ALPHA_VANTAGE_API_KEY"].strip()
sec_user_agent = os.environ["SEC_USER_AGENT"].strip()

out_dir = ROOT / "warehouse" / "lineage" / "architecture_v3_public_evidence_reconstruction_v1_20260919"
raw_dir = out_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
manifest = {
    "experiment_id": EXPERIMENT_ID,
    "design_signature": DESIGN_SIGNATURE,
    "qualification_dates": list(QUALIFICATION_DATES),
    "consumed_holdout_rows_read": 0,
    "models_fit": 0,
    "trades_or_orders": 0,
    "credentials_stored": False,
    "files": [],
}

def download(url, user_agent):
    req = urllib.request.Request(url, headers={"User-Agent": user_agent})
    with urllib.request.urlopen(req, timeout=90) as response:
        return response.read()

def save_raw(name, payload):
    path = raw_dir / name
    path.write_bytes(payload)
    manifest["files"].append({
        "path": str(path.relative_to(ROOT)),
        "bytes": len(payload),
        "sha256": hashlib.sha256(payload).hexdigest(),
    })
    return path

for request_number, (date, state_name) in enumerate(
    [(d, s) for d in QUALIFICATION_DATES for s in ("active", "delisted")]
):
    query = urllib.parse.urlencode({
        "function": "LISTING_STATUS", "date": date,
        "state": state_name, "apikey": api_key,
    })
    payload = download("https://www.alphavantage.co/query?" + query, "stockprediction2025-public-research/1.0")
    text = payload.decode("utf-8-sig", errors="replace").strip()
    assert text and not text.startswith("{"), f"Alpha Vantage did not return CSV for {date} {state_name}"
    rows = list(csv.DictReader(io.StringIO(text)))
    assert rows and {"symbol", "name", "exchange", "assetType", "ipoDate", "delistingDate", "status"}.issubset(rows[0])
    path = save_raw(f"alpha_vantage_listing_status_{date}_{state_name}.csv", payload)
    manifest["files"][-1]["rows"] = len(rows)
    if request_number < 5:
        time.sleep(13)

sec_payload = download("https://www.sec.gov/files/company_tickers_exchange.json", sec_user_agent)
json.loads(sec_payload.decode("utf-8"))
save_raw("sec_company_tickers_exchange.json", sec_payload)
manifest["collected_at_utc"] = datetime.now(timezone.utc).isoformat()
(out_dir / "collection_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps({
    "status": "bounded_sources_collected",
    "files": len(manifest["files"]),
    "output_dir": str(out_dir.relative_to(ROOT)),
    "holdout_rows_read": 0,
}, indent=2))


{
  "status": "bounded_sources_collected",
  "files": 7,
  "output_dir": "warehouse/lineage/architecture_v3_public_evidence_reconstruction_v1_20260919",
  "holdout_rows_read": 0
}


In [9]:
import pandas as pd

DATES = ["2021-08-23", "2024-01-02", "2026-05-28"]
PROBES = ["AAPL", "MSFT", "ABNB", "FB", "META", "SNE", "SONY", "ATVI", "TWTR", "BBBY", "BBBYQ"]
frames = {}
summary = {}
for d in DATES:
    for state in ("active", "delisted"):
        p = raw_dir / f"alpha_vantage_listing_status_{d}_{state}.csv"
        df = pd.read_csv(p, dtype=str).fillna("")
        df["symbol_norm"] = df["symbol"].str.upper()
        frames[(d, state)] = df
        exact_dupes = int(df.duplicated(["symbol"]).sum())
        case_collisions = int((df.groupby("symbol_norm")["symbol"].nunique() > 1).sum())
        summary[f"{d}_{state}"] = {"rows": int(len(df)), "columns": list(df.columns[:-1]), "exact_symbol_duplicates": exact_dupes, "case_collisions": case_collisions}

probe_rows = []
for d in DATES:
    for state in ("active", "delisted"):
        df = frames[(d, state)]
        hit = df[df["symbol_norm"].isin(PROBES)]
        for _, r in hit.iterrows():
            probe_rows.append({k: r.get(k, "") for k in ["symbol", "name", "exchange", "assetType", "ipoDate", "delistingDate", "status"]} | {"snapshot": d, "requested_state": state})

# Detect same normalized symbol attached to different names across the six snapshots.
all_rows = pd.concat([df.assign(snapshot=d, requested_state=s) for (d,s),df in frames.items()], ignore_index=True)
name_counts = all_rows.groupby("symbol_norm")["name"].nunique()
reuse_candidates = sorted(name_counts[name_counts > 1].index.tolist())
print(json.dumps({
    "snapshot_summary": summary,
    "probe_rows": probe_rows,
    "symbols_with_multiple_names_count": len(reuse_candidates),
    "symbols_with_multiple_names_sample": reuse_candidates[:25],
    "available_fields": sorted(set(all_rows.columns) - {"symbol_norm", "snapshot", "requested_state"}),
}, indent=2))


{
  "snapshot_summary": {
    "2021-08-23_active": {
      "rows": 10526,
      "columns": [
        "symbol",
        "name",
        "exchange",
        "assetType",
        "ipoDate",
        "delistingDate",
        "status"
      ],
      "exact_symbol_duplicates": 85,
      "case_collisions": 0
    },
    "2021-08-23_delisted": {
      "rows": 5130,
      "columns": [
        "symbol",
        "name",
        "exchange",
        "assetType",
        "ipoDate",
        "delistingDate",
        "status"
      ],
      "exact_symbol_duplicates": 23,
      "case_collisions": 0
    },
    "2024-01-02_active": {
      "rows": 10591,
      "columns": [
        "symbol",
        "name",
        "exchange",
        "assetType",
        "ipoDate",
        "delistingDate",
        "status"
      ],
      "exact_symbol_duplicates": 58,
      "case_collisions": 0
    },
    "2024-01-02_delisted": {
      "rows": 8091,
      "columns": [
        "symbol",
        "name",
        "exchange",
  

In [10]:
sec_doc = json.loads((raw_dir / "sec_company_tickers_exchange.json").read_text())
sec_fields = sec_doc.get("fields", [])
sec_data = pd.DataFrame(sec_doc.get("data", []), columns=sec_fields)

case_symbols = ["ARM", "CAVA", "FB", "META"]
case_rows = []
for d in DATES:
    for state in ("active", "delisted"):
        df = frames[(d, state)]
        for _, r in df[df["symbol_norm"].isin(case_symbols)].iterrows():
            case_rows.append({"snapshot": d, "requested_state": state, **{k:r.get(k,"") for k in ["symbol","name","exchange","assetType","ipoDate","delistingDate","status"]}})

bankruptcy_pattern = r"Bed Bath|WeWork|Revlon|Yellow Corporation|Party City"
bankruptcy_rows = all_rows[all_rows["name"].str.contains(bankruptcy_pattern, case=False, regex=True, na=False)][["snapshot","requested_state","symbol","name","ipoDate","delistingDate","status"]].drop_duplicates().to_dict("records")
duplicate_examples = all_rows[all_rows.duplicated(["snapshot","requested_state","symbol"], keep=False)][["snapshot","requested_state","symbol","name","exchange","assetType"]].head(30).to_dict("records")
print(json.dumps({
    "sec_fields": sec_fields,
    "sec_rows": int(len(sec_data)),
    "case_rows": case_rows,
    "bankruptcy_name_matches": bankruptcy_rows[:30],
    "duplicate_examples": duplicate_examples,
}, indent=2))


{
  "sec_fields": [
    "cik",
    "name",
    "ticker",
    "exchange"
  ],
  "sec_rows": 10438,
  "case_rows": [
    {
      "snapshot": "2021-08-23",
      "requested_state": "active",
      "symbol": "META",
      "name": "Meta Platforms Inc - Class A",
      "exchange": "NASDAQ",
      "assetType": "Stock",
      "ipoDate": "2012-05-18",
      "delistingDate": "",
      "status": "Active"
    },
    {
      "snapshot": "2024-01-02",
      "requested_state": "active",
      "symbol": "ARM",
      "name": "Arm Holdings plc.",
      "exchange": "NASDAQ",
      "assetType": "Stock",
      "ipoDate": "2023-09-14",
      "delistingDate": "",
      "status": "Active"
    },
    {
      "snapshot": "2024-01-02",
      "requested_state": "active",
      "symbol": "CAVA",
      "name": "Cava Group Inc",
      "exchange": "NYSE",
      "assetType": "Stock",
      "ipoDate": "2023-06-15",
      "delistingDate": "",
      "status": "Active"
    },
    {
      "snapshot": "2024-01-02",
      "r

In [11]:
result = {
  "schema_version": 1,
  "experiment_id": EXPERIMENT_ID,
  "design_signature": DESIGN_SIGNATURE,
  "design_fingerprint": fingerprint,
  "completed_at_utc": datetime.now(timezone.utc).isoformat(),
  "qualification_dates": DATES,
  "decision": "fail_stop_before_full_reconstruction",
  "architecture_v3_status": "blocked_by_historical_lineage",
  "consumed_holdout_rows_read": 0,
  "models_fit": 0,
  "trades_or_orders": 0,
  "credentials_stored": False,
  "evidence": {
    "files_collected": 7,
    "alpha_vantage_snapshot_rows": {k:v["rows"] for k,v in summary.items()},
    "alpha_vantage_exact_symbol_duplicates": {k:v["exact_symbol_duplicates"] for k,v in summary.items()},
    "case_normalization_collisions": {k:v["case_collisions"] for k,v in summary.items()},
    "sec_current_rows": int(len(sec_data)),
    "sec_fields": sec_fields,
    "continuous_listing_examples": ["AAPL", "MSFT"],
    "new_listing_examples": {"ABNB":"2020-12-10", "ARM":"2023-09-14", "CAVA":"2023-06-15"},
    "acquisition_delisting_examples": {"ATVI":"2023-10-13", "TWTR":"2022-10-28"},
    "historical_mapping_failure": "The 2021-08-23 active snapshot uses META for Meta Platforms even though the issuer traded as FB on that date; the file backfills a later ticker rather than preserving the historical ticker.",
    "ticker_reuse_failure": "The 2026-05-28 active snapshot assigns FB to a ProShares ETF with IPO date 2025-06-26, while the 2021 Meta issuer is retroactively labeled META. Symbol alone cannot distinguish the two securities.",
    "duplicate_identity_failure": "Within-date exact symbols map to different names/entities (for example ARIS, B, BEST, CADE and CNR), so symbol plus date is not a unique security key.",
    "bankruptcy_event_limit": "Name searches found records such as WeWork-related instruments, but the files provide no bankruptcy/event reason or cash/terminal-value field; Bed Bath naming also reflects later issuer/name reuse and is not a trustworthy bankruptcy lineage."
  },
  "capability_gates": {
    "historical_membership": "fail_not_point_in_time_trustworthy",
    "stable_security_identity": "fail_no_identifier_in_alpha_vantage_and_current_only_sec_snapshot",
    "historical_ticker_mappings": "fail_retroactive_symbol_backfill_observed",
    "listing_dates": "partial_ipoDate_present_but_identity_ambiguous",
    "delisting_events": "partial_date_and_status_present_but_no_event_reason",
    "adjusted_and_unadjusted_prices": "not_qualified_by_these_sources",
    "split_and_dividend_handling": "not_qualified_by_these_sources",
    "terminal_or_delisting_values": "fail_absent",
    "merger_acquisition_classification": "partial_delisting_date_only",
    "bankruptcy_classification": "fail_absent",
    "ticker_reuse": "fail_explicit_FB_reuse_observed_without_stable_identity",
    "case_normalization_collisions": "pass_none_observed_but_does_not_resolve_exact_symbol_collisions"
  },
  "stop_reason": "At least one mandatory qualification gate failed; full 756-date reconstruction is prohibited.",
  "permitted_future_use": "Discovery/candidate-event evidence only. Do not use these snapshots as the canonical point-in-time universe or identity map."
}
result_path = ROOT / "research_context" / "architecture_v3_public_evidence_reconstruction_result_v1_20260919.json"
result_path.write_text(json.dumps(result, indent=2) + "\n")
(out_dir / "qualification_summary.json").write_text(json.dumps(result, indent=2) + "\n")

md = f"""# Architecture v3 public-evidence reconstruction qualification

**Decision:** FAIL — stop before full reconstruction.  
**Dates tested:** {', '.join(DATES)}  
**Holdout rows read:** 0  
**Models fit / trades:** 0 / 0

Alpha Vantage returned all six requested dated listing-status files and SEC returned one current CIK/ticker file. The source still fails the mandatory lineage gates:

- The 2021-08-23 snapshot labels Meta Platforms as META, although the issuer traded as FB then. This is retroactive ticker backfilling, not a trustworthy historical mapping.
- On 2026-05-28, FB belongs to a ProShares ETF with IPO date 2025-06-26. Without a stable identifier, this ticker reuse is indistinguishable from Meta's former ticker.
- Exact duplicate symbols occur within every snapshot/state file; examples include symbols attached to different company names. Counts range from 23 to 239 per file.
- Alpha Vantage supplies only symbol, name, exchange, asset type, IPO date, delisting date, and status. It supplies no stable security identifier, event reason, bankruptcy classification, or terminal/delisting value.
- The SEC file supplies current CIK/name/ticker/exchange rows only; it does not turn the Alpha Vantage snapshots into a historical identifier map.
- Adjusted/unadjusted prices and split/dividend handling were not qualified by these two sources. No further calls were made after the identity gate failed.

Useful but insufficient evidence was observed: AAPL/MSFT continuous listings; ABNB/ARM/CAVA IPO dates; ATVI/TWTR delisting dates; and no case-only normalization collisions. These do not cure the identity and terminal-return failures.

Architecture v3 remains blocked. The collected files may be used only for candidate discovery and manual corroboration, never as the canonical point-in-time universe.
"""
md_path = ROOT / "research_context" / "architecture_v3_public_evidence_reconstruction_result_v1_20260919.md"
md_path.write_text(md)

gate_doc = json.loads(gate_path.read_text())
entry = next(x for x in gate_doc["next_experiments"] if x.get("experiment_id") == EXPERIMENT_ID)
entry.update({
  "status": "completed_source_qualification_failed_do_not_repeat",
  "decision": result["decision"],
  "result_artifact": str(result_path.relative_to(ROOT)),
  "holdout_rows_read": 0,
  "full_reconstruction_authorized": False
})
gate_path.write_text(json.dumps(gate_doc, indent=2) + "\n")

state_path = ROOT / "research_context" / "current_research_state_v1.json"
state_doc = json.loads(state_path.read_text())
state_doc["architecture_v3_public_evidence_reconstruction"] = {
  "status": entry["status"],
  "decision": result["decision"],
  "result_artifact": str(result_path.relative_to(ROOT)),
  "critical_findings": [result["evidence"]["historical_mapping_failure"], result["evidence"]["ticker_reuse_failure"], result["evidence"]["duplicate_identity_failure"]],
  "full_reconstruction_authorized": False,
  "consumed_holdout_rows_read": 0
}
state_doc["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
state_path.write_text(json.dumps(state_doc, indent=2) + "\n")
print(json.dumps({"decision": result["decision"], "result": str(result_path.relative_to(ROOT)), "markdown": str(md_path.relative_to(ROOT)), "holdout_rows_read": 0, "full_reconstruction_authorized": False}, indent=2))


{
  "decision": "fail_stop_before_full_reconstruction",
  "result": "research_context/architecture_v3_public_evidence_reconstruction_result_v1_20260919.json",
  "markdown": "research_context/architecture_v3_public_evidence_reconstruction_result_v1_20260919.md",
  "holdout_rows_read": 0,
  "full_reconstruction_authorized": false
}


In [12]:
import subprocess

def git_out(*args):
    return subprocess.run(["git", *args], cwd=ROOT, text=True, capture_output=True, check=False)

for args in [("status", "--short", "--branch"), ("remote", "-v"), ("log", "-1", "--oneline", "--decorate")]:
    cp = git_out(*args)
    print("$ git", " ".join(args))
    print((cp.stdout or cp.stderr).strip())


$ git status --short --branch
## main...origin/main
 M public/data/latest-analysis.json
 M research_context/context_gate.json
 M research_context/end_to_end_pipeline_map_v1.json
 M research_context/stock_dependency_architecture_v1.md
 M scripts/build_polygon_lineage_coverage_probe.py
 M scripts/context_gate.py
?? .ipynb_checkpoints/
?? artifacts/graph_signal_confirmatory_v1/pid
?? artifacts/graph_signal_confirmatory_v1/run.log
?? artifacts/graph_signal_confirmatory_v1/v2_pid
?? artifacts/graph_signal_confirmatory_v1/v2_run.log
?? artifacts/graph_signal_history400_v1/raw_features.csv
?? artifacts/point_in_time_residual_panel_5y_v1/
?? artifacts/purged_walk_forward_residual_baselines_v1/
?? artifacts/regime_exposure_policy_v1.run.log
?? artifacts/regime_exposure_policy_v1/
?? artifacts/regime_exposure_policy_v1_repro_a588f5a/
?? artifacts/residual_sleeve_family_confirmation_v1/
?? artifacts/similarity_weighted_linear_400d_v1/.ipynb_checkpoints/
?? artifacts/similarity_weighted_linear_400